## 01_clean_churn_imported_data
MySQL data ingestion for the telecom churn project (credentials loaded from .env, no hardcoded password).

In [ ]:
# --- Cell 1: Imports ---
import os
from dotenv import load_dotenv
import mysql.connector
from sqlalchemy import create_engine
import pandas as pd

In [ ]:
# --- Cell 2: Load your database password safely (instead of typing it in) ---
# Before running this, create a file named ".env" in your project folder
# (same folder level as the notebook folder) with these 4 lines, and put
# YOUR real password there:
#
#   DB_HOST=localhost
#   DB_USER=root
#   DB_PASSWORD=your_actual_password
#   DB_NAME=customer_db
#
# Then add a line saying ".env" to your .gitignore file, so it never gets
# uploaded to GitHub. This keeps your real password private, even though
# your code is public.
load_dotenv()

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_USER = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_NAME = os.getenv("DB_NAME", "customer_db")

In [ ]:
# --- Cell 3: Connect to MySQL ---
mydb = mysql.connector.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD
)

cursor = mydb.cursor()

In [ ]:
# --- Cell 4: Create the database if it doesn't already exist ---
cursor.execute(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
print("✅ Database created successfully!")

In [ ]:
# --- Cell 5: SQLAlchemy engine (needed to load data into MySQL easily) ---
engine = create_engine(f"mysql+mysqlconnector://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}")

In [ ]:
# --- Cell 6: Load your cleaned CSV file ---
# A relative path (instead of a full "C:\Users\..." path) means this
# notebook will also work on anyone else's computer, including a
# recruiter who clones your repo. Keep clean_churn.csv inside a "data"
# folder in your project.
csv_file_path = "../data/clean_churn.csv"

df = pd.read_csv(csv_file_path)
print(df.head())

In [ ]:
# --- Cell 7: Save the data into your MySQL table ---
table_name = "churn_data"
df.to_sql(table_name, con=engine, if_exists="replace", index=False)
print(f"✅ {len(df)} rows saved to the '{table_name}' table")

In [ ]:
# --- Cell 8: Double-check the table was created ---
cursor.execute(f"USE {DB_NAME}")
cursor.execute("SHOW TABLES")
tables = cursor.fetchall()
print("Tables in DB:", tables)

In [ ]:
# --- Cell 9 (NEW): Close the connection when you're done ---
cursor.close()
mydb.close()
print("✅ Connection closed")

# Note: the old cell that dropped a database called "churn_db" has been
# removed on purpose. That database was never created anywhere in this
# notebook — it was leftover code from an earlier attempt and wasn't
# doing anything, so it's cleaner to just take it out.